# Runbook mensuel — Pipeline Data & Décision Marketing

**Awalé Boissons**

**Objectif :** produire chaque mois une analyse reproductible du budget marketing, des ventes et de la Customer Voice, puis préparer une proposition testable d'allocation budgétaire en moins d'une heure, sans data engineer.


## 1. Préparer les nouvelles données

Déposer les exports du nouveau mois sans modifier les historiques.

```
data/
  raw/
    awale_boissons_starter_dataset.xlsx
```

**Règle :** une valeur manquante reste manquante. Une journée absente n'est jamais transformée automatiquement en zéro vente.


## 2. Contrôler DuckDB

Vérifier que les tables RAW attendues sont présentes et que le nouveau périmètre est bien chargé.

```bash
cd /mnt/c/Users/KSOMS/Favorites/awale_boissons
duckdb data/awale.duckdb
SHOW TABLES;
```


## 3. Exécuter dbt

Lancer le pipeline puis les tests.

```bash
cd dbt
dbt run
dbt test
```

**Critère de passage :** aucune erreur et aucun avertissement.


## 4. Contrôles de qualité des données

| Domaine | Contrôles essentiels |
|---|---|
| Ventes | Doublons, dates, retours, unités/CA négatifs, jours manquants, couverture. |
| Marketing | Campagnes vs media plan vs facturé ; divergences visibles ; aucune vérité universelle choisie arbitrairement. |
| WhatsApp | Montants manquants, order_ref répétés, parsing des articles, produits inconnus. |
| Social | Doublons, unicité de comment_id, prédictions non NULL, volume source = volume prédit. |

**Important :** ne jamais construire un catalogue de prix à partir de `amount_fcfa` / `quantity`. Les montants peuvent contenir des anomalies ou agrégations.


## 5. Exécuter et surveiller l'IA

- Le benchmark humain de 50 commentaires reste séparé de l'inférence complète : il sert à évaluer le modèle, pas à prétendre à un fine-tuning.
- Comparer le nombre de commentaires source et le nombre de prédictions.
- Vérifier l'unicité de `comment_id` et l'absence de NULL sur `language`, `sentiment`, `theme`, `product` et `spam`.
- Conserver version du modèle, résultats d'évaluation, erreurs, volume, temps d'exécution, coût éventuel et limites.
- Ne jamais laisser le modèle inventer des chiffres ou des faits absents des données.


## 6. Vérifier le dashboard — 4 blocs maximum

| Bloc | Question |
|---|---|
| 1. Où va l'argent ? | Répartition du budget et divergences avec le plan. |
| 2. Ventes | Évolution du CA et des unités, avec couverture et jours manquants. |
| 3. Customer Voice | Sentiment, thèmes et produits dans les commentaires. |
| 4. What to do next | Proposition de budget testable, conditions de mesure et limites. |


## 7. Chaîne de décision

```
Faits observés → qualité des données → observations → signaux → limites → hypothèses → test → décision
```

**Exemple :** beaucoup de commentaires négatifs sur le prix constitue un signal de Customer Voice. Cela ne démontre pas qu'un canal marketing en est la cause ni qu'un changement de budget améliorera les ventes.


## 8. Règles méthodologiques non négociables

- Ne pas transformer les valeurs manquantes en zéros.
- Ne pas inventer de produit, format ou prix.
- Ne pas présenter une association canal → ventes comme une attribution causale.
- Ne pas utiliser `mart_channel_performance_monthly` pour attribuer le CA aux canaux : le CA mensuel total y est répété par canal.
- Ne pas transformer CPC, CPM, impressions ou clics en preuve d'efficacité commerciale.
- Ne pas présenter l'allocation de 15 M FCFA comme un ROI causal.
- Documenter les conditions d'instrumentation nécessaires pour tester les canaux faiblement mesurés.


## 9. Proposition de séquence pour les 15 M FCFA

| Canal | Budget proposé |
|---|---|
| Meta | 5 000 000 FCFA |
| TikTok | 4 000 000 FCFA |
| Google | 2 500 000 FCFA |
| Radio | 1 500 000 FCFA |
| Influenceurs | 1 000 000 FCFA |
| Activation terrain | 1 000 000 FCFA |
| **TOTAL** | **15 000 000 FCFA** |

Cette proposition est un budget de test et d'instrumentation fondé sur les données observées et leurs limites ; elle ne constitue pas un classement causal des canaux.


## 10. Conditions de test par canal

| Canal | Condition de mesure |
|---|---|
| Meta | Impressions, clics et conversion mesurable. |
| TikTok | Impressions, clics et conversion mesurable. |
| Google | Clics et conversion mesurable. |
| Radio | Code, numéro ou mécanisme de suivi dédié. |
| Influenceurs | Lien, code ou mécanisme de suivi dédié. |
| Activation terrain | Réconcilier facturation et dépenses campagne avant extrapolation. |


## 11. Cible de temps mensuelle

| Étape | Cible |
|---|---|
| Préparation fichiers | 10 min |
| Chargement DuckDB | 5 min |
| dbt run | 5 min |
| dbt test | 5 min |
| Contrôles qualité | 10 min |
| Inférence IA | 10–15 min |
| Contrôle dashboard | 5 min |
| **TOTAL** | **≈ 50–55 min** |


## 12. Commandes de fin de run

```bash
cd /mnt/c/Users/KSOMS/Favorites/awale_boissons
# charger les nouvelles données
cd dbt
dbt run
dbt test
cd ..
streamlit run app/app.py
```

**Livrable mensuel :** pipeline exécuté et testé, dashboard actualisé, contrôles qualité vérifiés, proposition budgétaire documentée et limites explicitement signalées.
